In [ ]:
import os
from pathlib import Path
import yaml
from ultralytics import YOLO
import logging
from collections import Counter

# # Set up logging
# logger = logging.getLogger()
# logger.setLevel(logging.INFO)
# stream_handler = logging.StreamHandler()
# stream_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
# logger.addHandler(stream_handler)
# file_handler = logging.FileHandler("/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/training_v3.log")
# file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
# logger.addHandler(file_handler)

# Paths
balanced_dataset_dir = Path("/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/balanced_dataset")
data_yaml = balanced_dataset_dir / "data.yaml"
runs_dir = Path("/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs")
runs_dir.mkdir(parents=True, exist_ok=True)

# # Verify dataset structure
# logger.info(f"Checking balanced_dataset structure: {list(balanced_dataset_dir.glob('*'))}")
# for split in ['train', 'val']:
#     img_count = len(list((balanced_dataset_dir / split / " afectados").glob('*.jpg')))
#     logger.info(f"{split} images: {img_count}")

# # Check class balance
# def count_classes(label_dir):
#     class_counts = Counter()
#     for label_file in label_dir.glob('*.txt'):
#         with open(label_file, 'r') as f:
#             for line in f:
#                 class_id = int(line.split()[0])
#                 class_counts[class_id] += 1
#     return class_counts

# train_labels = balanced_dataset_dir / 'train' / 'labels'
# val_labels = balanced_dataset_dir / 'val' / 'labels'
# logger.info("Train class counts: %s", count_classes(train_labels))
# logger.info("Val class counts: %s", count_classes(val_labels))

# # Read class names from data.yaml
# with open(data_yaml, 'r') as f:
#     data_config = yaml.safe_load(f)
# class_names = data_config['names']

# Hyperparameters (best config from previous tuning)
lr0 = 0.001
batch = 16
imgsz = 640
scale = 0.2
mosaic = 0.5
epochs = 20

# Model
model = YOLO("yolo11n.pt")  # Loads pre-trained YOLO11n from Ultralytics

# Run name
run_name = "mps"
run_dir = runs_dir / run_name

# # Train with enhanced augmentation and regularization
# logger.info(f"Starting training: {run_name}")
results = model.train(
    data=str(data_yaml),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    lr0=lr0,
    warmup_epochs=3,
    device='mps',
    name=run_name,
    project=str(runs_dir),
    patience=10,
    verbose=True,
    plots=True,
    augment=True,
    mosaic=mosaic,  # Grid-searched value
    mixup=0,
    degrees=15,  # Increased for robustness
    translate=0.1,  # Slightly more translation
    scale=scale,  # Grid-searched value
    shear=0.0,
    hsv_h=0.015,  # Stronger hue adjustment
    hsv_s=0.7,  # Stronger saturation
    hsv_v=0.4,  # Stronger value
    dropout=0.1,  # Add dropout for regularization
    weight_decay=0.0005,  # Add weight decay
    
    conf = 0.6,      # Higher confidence threshold (default: 0.25)
    iou = 0.7,       # Higher IoU threshold (default: 0.45)
    # max_det = 100,   # Fewer max detections (default: 300)
)

model.export(format="coreml", nms=True)

# # Load best model and evaluate
# best_model_path = run_dir / "weights/best.pt"
# if not best_model_path.exists():
#     logger.error(f"Best model not found at {best_model_path}")
# else:
#     best_model = YOLO(str(best_model_path))
#     val_results = best_model.val(data=str(data_yaml), split='val')
#     avg_recall = val_results.box.r.mean()

#     # Log per-class recall and flag low performers
#     logger.info(f"Validation results for {run_name}:")
#     for i, cls in enumerate(class_names):
#         recall = val_results.box.r[i]
#         logger.info(f"Class {cls}: Recall={recall:.4f}")
#         if recall < 0.7:
#             logger.warning(f"Low recall for {cls}: {recall:.4f}")
#     logger.info(f"Average recall: {avg_recall:.4f}")

#     # Export to CoreML
#     logger.info("Exporting to CoreML format...")

#     coreml_path = best_model.export(format="coreml", nms=True)
#     logger.info(f"Export completed. CoreML model saved at: {coreml_path}")

New https://pypi.org/project/ultralytics/8.3.154 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.130 🚀 Python-3.9.6 torch-2.5.1 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=0.6, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/balanced_dataset/data.yaml, degrees=15, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=yolo11n_

train: Scanning /Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/balanced_dataset/train/labels.cache... 3711 images, 1004 backgrounds, 0 corrupt: 100%|██████████| 3711/3711 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 42, len(boxes) = 4101. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 637.0±412.4 MB/s, size: 168.8 KB)



val: Scanning /Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/balanced_dataset/val/labels.cache... 751 images, 245 backgrounds, 0 corrupt: 100%|██████████| 751/751 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 2, len(boxes) = 860. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
Plotting labels to /Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/labels.jpg... 


...

scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.5.1 has not been tested with coremltools. You may run into unexpected errors. Torch 2.5.0 is the most recent version that has been tested.



CoreML: starting export with coremltools 8.3.0...


Tuple detected at graph output. This will be flattened in the converted model.
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 163.57 passes/s]


CoreML Pipeline: starting pipeline with coremltools 8.3.0...
CoreML Pipeline: pipeline success
CoreML: export success ✅ 6.1s, saved as '/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/weights/best.mlpackage' (5.1 MB)

Export complete (6.2s)
Results saved to /Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/weights
Predict:         yolo predict task=detect model=/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/weights/best.mlpackage imgsz=640  
Validate:        yolo val task=detect model=/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/weights/best.mlpackage imgsz=640 data=/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/h

'/Users/jasper/Anna/Uni/UTS/MAI/Subjects/AIS/showcase/models/hazard/yolo11/runs/yolo11n_balanced_lr0.001_b16_scale0.2_mosaic0.5_coreml_v3_8/weights/best.mlpackage'